In [7]:
import pandas as pd
import numpy as np

# ==============================================================================
# ASSIGNMENT 20: Building & Deploying a Recommendation System
# Dataset: TMDB 5000 Movie Dataset (tmdb_5000_movies.csv)

# 1. Load the dataset using Pandas
df = pd.read_csv("tmdb_5000_movies.csv")

# 2. Print shape, column names, first 5 rows, and essential details
print("--- 1. DATASET SHAPE ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n--- 2. COLUMN NAMES ---")
print(df.columns.tolist())

print("\n--- 3. FIRST 5 ROWS ---")
print(df[['id', 'title', 'genres', 'overview', 'vote_average']].head())

print("\n--- 4. ESSENTIAL DETAILS (MISSING VALUES & DUPLICATES) ---")
print(df[['title', 'overview', 'genres', 'keywords', 'tagline']].isnull().sum())
print(f"Duplicate movie titles: {df['title'].duplicated().sum()}")

# Drop duplicate titles and reset index so similarity matrix indices match DataFrame rows
df = df.drop_duplicates(subset='title').reset_index(drop=True)

# 3. Identify text column(s) used for recommendations
print("\n--- 5. COLUMNS SELECTED FOR RECOMMENDATION ---")
print("Item Identifier : 'title'")
print("Text Features   : 'overview', 'genres', 'keywords', 'tagline'")

--- 1. DATASET SHAPE ---
Rows: 4803, Columns: 20

--- 2. COLUMN NAMES ---
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

--- 3. FIRST 5 ROWS ---
       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id

In [10]:
import re
import json
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ==============================================================================
# PART 1 — Data Preprocessing
# Task 2: Text Preprocessing for Recommendation
# ==============================================================================

stop_words = set(ENGLISH_STOP_WORDS)

# Helper to parse JSON-formatted metadata columns ('genres' and 'keywords')
def extract_names(json_str):
    if pd.isna(json_str) or not isinstance(json_str, str):
        return ""
    try:
        items = json.loads(json_str)
        return " ".join([item['name'].replace(" ", "") for item in items])
    except Exception:
        return ""

# 4. Handle missing values (replace with empty string) & combine text features
df['overview'] = df['overview'].fillna("")
df['tagline'] = df['tagline'].fillna("")
df['genres_clean'] = df['genres'].apply(extract_names)
df['keywords_clean'] = df['keywords'].apply(extract_names)

df['raw_text'] = (
    df['overview'] + " " +
    df['genres_clean'] + " " +
    df['keywords_clean'] + " " +
    df['tagline']
)

def clean_text_pipeline(text):
    if not isinstance(text, str):
        return ""
    # 1. Convert text to lowercase
    text = text.lower()
    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 3. Remove stopwords and single-character tokens
    tokens = [word for word in text.split() if word not in stop_words and len(word) > 1]
    return " ".join(tokens)

# Store cleaned text in a new column: clean_text
df['clean_text'] = df['raw_text'].apply(clean_text_pipeline)

print("--- SAMPLE CLEANED TEXT (FIRST 3 MOVIES) ---")
for idx in range(3):
    print(f"\nMovie Title : {df.loc[idx, 'title']}")
    print(f"clean_text  : {df.loc[idx, 'clean_text'][:180]}...")

--- SAMPLE CLEANED TEXT (FIRST 3 MOVIES) ---

Movie Title : Avatar
clean_text  : nd century paraplegic marine dispatched moon pandora unique mission torn following orders protecting alien civilization action adventure fantasy sciencefiction cultureclash future ...

Movie Title : Pirates of the Caribbean: At World's End
clean_text  : captain barbossa long believed dead come life headed edge earth turner elizabeth swann quite adventure fantasy action ocean drugabuse exoticisland eastindiatradingcompany loveofone...

Movie Title : Spectre
clean_text  : cryptic message bond past sends trail uncover sinister organization battles political forces secret service alive bond peels layers deceit reveal terrible truth spectre action adve...


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ==============================================================================
# PART 2 — Text Vectorization
# Task 3: Vectorization using TF-IDF
# ==============================================================================

# 1 & 2. Initialize TfidfVectorizer with max_features and ngram_range
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Convert clean_text into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

# 3. Display shape of TF-IDF matrix
print("--- TF-IDF MATRIX DETAILS ---")
print(f"Shape of TF-IDF Matrix (Movies x Features): {tfidf_matrix.shape}")
print(f"Total Vocabulary Features                 : {len(tfidf.get_feature_names_out())}")
print(f"Sample Features                           : {tfidf.get_feature_names_out()[:15].tolist()}")

--- TF-IDF MATRIX DETAILS ---
Shape of TF-IDF Matrix (Movies x Features): (4800, 5000)
Total Vocabulary Features                 : 5000
Sample Features                           : ['aaron', 'abandoned', 'abducted', 'abilities', 'ability', 'able', 'aboard', 'abuse', 'abusive', 'academic', 'academy', 'accept', 'accepted', 'accepts', 'access']


In [19]:
# ==============================================================================
# PART 3 — Recommendation Logic
# Task 5: Build Recommendation Function
# ==============================================================================

# Create case-insensitive title-to-index lookup
indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

def recommend(item_name, top_n=5):
    """Returns top N similar items based on content cosine similarity."""
    query = item_name.strip().lower()
    if query not in indices:
        return f"Item '{item_name}' not found in dataset."
    
    # 1. Find index of the selected item
    idx = indices[query]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
        
    # 2. Compute similarity scores for all items against the selected item
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # 3. Sort descending by similarity score and exclude the selected item itself
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1 : top_n + 1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores = [round(float(i[1]), 4) for i in sim_scores]
    
    return pd.DataFrame({
        'Recommended Movie': df['title'].iloc[movie_indices].values,
        'Similarity Score': scores
    })

# Test with 3 different items
sample_movies = ["Avatar", "The Dark Knight", "Toy Story"]

for movie in sample_movies:
    print(f"\n=== Top 5 Recommendations for '{movie}' ===")
    print(recommend(movie, top_n=5).to_string(index=False))


=== Top 5 Recommendations for 'Avatar' ===
                         Recommended Movie  Similarity Score
Lara Croft Tomb Raider: The Cradle of Life            0.2221
                                Titan A.E.            0.1791
                       Battle: Los Angeles            0.1738
                             Falcon Rising            0.1726
                                  Galaxina            0.1662

=== Top 5 Recommendations for 'The Dark Knight' ===
                      Recommended Movie  Similarity Score
                  The Dark Knight Rises            0.4401
                         Batman Returns            0.3735
                          Batman Begins            0.3359
                         Batman Forever            0.3250
Batman: The Dark Knight Returns, Part 2            0.3049

=== Top 5 Recommendations for 'Toy Story' ===
     Recommended Movie  Similarity Score
           Toy Story 2            0.4699
           Toy Story 3            0.4573
The 40 Year Old Vir

In [22]:
# ==============================================================================
# PART 4 & 5 — Generate app.py (Streamlit), requirements.txt, and README.md
# ==============================================================================

app_code = '''import re
import json
import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(page_title="Movie Recommendation System", page_icon="🎬", layout="centered")

STOP_WORDS = set(ENGLISH_STOP_WORDS)

def extract_names(json_str):
    if pd.isna(json_str) or not isinstance(json_str, str):
        return ""
    try:
        items = json.loads(json_str)
        return " ".join([item["name"].replace(" ", "") for item in items])
    except Exception:
        return ""

def clean_text_pipeline(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", " ", text)
    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) > 1]
    return " ".join(tokens)

@st.cache_data
def build_recommender():
    df = pd.read_csv("tmdb_5000_movies.csv")
    df = df.drop_duplicates(subset="title").reset_index(drop=True)
    
    df["overview"] = df["overview"].fillna("")
    df["tagline"] = df["tagline"].fillna("")
    df["genres_clean"] = df["genres"].apply(extract_names)
    df["keywords_clean"] = df["keywords"].apply(extract_names)
    
    combined = (
        df["overview"] + " " +
        df["genres_clean"] + " " +
        df["keywords_clean"] + " " +
        df["tagline"]
    )
    df["clean_text"] = combined.apply(clean_text_pipeline)
    
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    tfidf_matrix = tfidf.fit_transform(df["clean_text"])
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
    return df, cosine_sim

df, cosine_sim = build_recommender()

def recommend(item_name, top_n=5):
    idx = df[df["title"] == item_name].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1 : top_n + 1]
    
    results = []
    for movie_idx, score in sim_scores:
        results.append({
            "Movie Title": df.loc[movie_idx, "title"],
            "Rating": df.loc[movie_idx, "vote_average"],
            "Similarity": f"{score:.2%}",
            "Overview": df.loc[movie_idx, "overview"][:160] + "..."
        })
    return pd.DataFrame(results)

# --- Simple Streamlit UI ---
st.title("🎬 Content-Based Movie Recommendation System")
st.write("Select a movie to generate recommendations based on **TF-IDF** and **Cosine Similarity**.")

selected_movie = st.selectbox("Select a Movie:", sorted(df["title"].tolist()))
top_n = st.slider("Number of Recommendations:", min_value=3, max_value=10, value=5)

if st.button("Get Recommendations"):
    recommendations = recommend(selected_movie, top_n=top_n)
    st.subheader(f"Top {top_n} Recommended Movies for '{selected_movie}'")
    st.dataframe(recommendations, use_container_width=True, hide_index=True)
'''

requirements_txt = """streamlit
pandas
numpy
scikit-learn
"""

readme_md = """# Content-Based Movie Recommendation System

An end-to-end content-based recommendation system built using Python, Pandas, Scikit-learn, and Streamlit, and deployed on Render.

## Dataset
- **Dataset Name:** TMDB 5000 Movie Dataset (`tmdb_5000_movies.csv`)
- **Kaggle Link:** https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

## Features
- Text cleaning pipeline (lowercasing, punctuation removal, stopword removal, null handling)
- TF-IDF vectorization (`max_features=5000`, `ngram_range=(1, 2)`)
- Pairwise Cosine Similarity matrix
- Interactive Streamlit web interface

## Run Locally
```bash
pip install -r requirements.txt
streamlit run app.py

SyntaxError: incomplete input (432480385.py, line 92)

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

# ==============================================================================
# PART 2 — Text Vectorization
# Task 4: Similarity Computation
# ==============================================================================

# 1 & 2. Compute cosine similarity between all items and store similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("--- COSINE SIMILARITY MATRIX ---")
print(f"Similarity Matrix Shape: {cosine_sim.shape}")
print("\nSample 5x5 Sub-matrix:")
print(np.round(cosine_sim[:5, :5], 4))

# 3. Explain briefly why cosine similarity is used
print("\n--- WHY COSINE SIMILARITY IS USED ---")
print("""1. Length Normalization: Movie descriptions vary in word count. Cosine similarity measures the angle between two TF-IDF vectors rather than their Euclidean magnitude, so longer overviews do not unfairly overpower shorter ones.
2. Bounded Scale [0, 1]: Produces clean similarity scores between 0 (no shared content) and 1 (identical content), making item ranking straightforward.
3. High-Dimensional Efficiency: Computes fast dot products across sparse 5,000-dimensional TF-IDF vectors.""")

--- COSINE SIMILARITY MATRIX ---
Similarity Matrix Shape: (4800, 4800)

Sample 5x5 Sub-matrix:
[[1.     0.0514 0.0233 0.0192 0.1005]
 [0.0514 1.     0.0191 0.0033 0.0524]
 [0.0233 0.0191 1.     0.0086 0.0379]
 [0.0192 0.0033 0.0086 1.     0.0127]
 [0.1005 0.0524 0.0379 0.0127 1.    ]]

--- WHY COSINE SIMILARITY IS USED ---
1. Length Normalization: Movie descriptions vary in word count. Cosine similarity measures the angle between two TF-IDF vectors rather than their Euclidean magnitude, so longer overviews do not unfairly overpower shorter ones.
2. Bounded Scale [0, 1]: Produces clean similarity scores between 0 (no shared content) and 1 (identical content), making item ranking straightforward.
3. High-Dimensional Efficiency: Computes fast dot products across sparse 5,000-dimensional TF-IDF vectors.
